In [1]:
from pathlib import Path
import pandas as pd
import os
import re
from itertools import combinations
import json

In [3]:
data_dir = '../data/orig/shots_dataset/'
output_dir = '../data/merged/'

In [3]:
# Optional download raw dataset

#import kagglehub
#path = kagglehub.dataset_download(
#    "brains14482/nba-playbyplay-and-shotdetails-data-19962021",
#    output_dir= data_dir
#)

In [4]:
def parse_filename(fname):
    # Helper to extract year and playoffs
    m = re.match(r"shotdetail_(po_)?(\d{4})\.csv", fname)
    if m:
        is_po = m.group(1) is not None
        year = int(m.group(2))
        return year, is_po
    return None

In [5]:
tables2merge = ['nbastatsv3_', 'nbastats_'] # tables to merge with 'shotdetail_'

merge_keys = {
    'shotdetail_': ['GAME_ID', 'GAME_EVENT_ID'], 
    'nbastatsv3_': ['gameId', 'actionNumber'],
    'nbastats_': ['GAME_ID', 'EVENTNUM']
}

In [6]:
shot_files = [
    f for f in os.listdir(data_dir)
    if f.startswith("shotdetail_") and f.endswith(".csv")
]

shot_files = sorted(shot_files, key=lambda f: parse_filename(f))


In [8]:
import pandas as pd

df_merged = None
count_rows_addded = {}
duplicated_columns = {}

for shot_file in shot_files:
    year, is_po = parse_filename(shot_file)
    suffix = f"po_{year}" if is_po else f"{year}"

    df_year_merged = pd.read_csv(os.path.join(data_dir, shot_file))
    base_len = len(df_year_merged.index)

    # horizontal merges
    for tabletype in tables2merge:
        other_file = f"{tabletype}{suffix}.csv"
        path = os.path.join(data_dir, other_file)

        if not os.path.exists(path):
            continue

        df_other = pd.read_csv(path)

        df_year_merged = pd.merge(
            df_year_merged,
            df_other,
            how='left',
            left_on=merge_keys['shotdetail_'],
            right_on=merge_keys[tabletype]
        )

    # metadata
    df_year_merged["year"] = year
    df_year_merged["is_playoffs"] = is_po
    
    count_rows_addded[suffix] = len(df_year_merged.index) - base_len
    duplicated_columns[suffix] = [(i, j) for i,j in combinations(df_year_merged, 2) if df_year_merged[i].equals(df_year_merged[j])]

    # vertical merge
    if df_merged is None:
        df_merged = df_year_merged
    else:
        df_merged = pd.concat([df_merged, df_year_merged], ignore_index=True)



In [10]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 6454704 entries, 0 to 6454703
Data columns (total 83 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   GRID_TYPE                  str    
 1   GAME_ID                    int64  
 2   GAME_EVENT_ID              int64  
 3   PLAYER_ID                  int64  
 4   PLAYER_NAME                str    
 5   TEAM_ID                    int64  
 6   TEAM_NAME                  str    
 7   PERIOD_x                   int64  
 8   MINUTES_REMAINING          int64  
 9   SECONDS_REMAINING          int64  
 10  EVENT_TYPE                 str    
 11  ACTION_TYPE                str    
 12  SHOT_TYPE                  str    
 13  SHOT_ZONE_BASIC            str    
 14  SHOT_ZONE_AREA             str    
 15  SHOT_ZONE_RANGE            str    
 16  SHOT_DISTANCE              int64  
 17  LOC_X                      int64  
 18  LOC_Y                      int64  
 19  SHOT_ATTEMPTED_FLAG        int64  
 20  SHOT_MADE_FLA

# Save Metadata and combined table

In [12]:
df_merged.to_csv(Path(output_dir) / "merged_dataset.csv", index=False)

meta_data = {
    "count_rows_added": count_rows_addded,
    "duplicated_columns": duplicated_columns
}

with open(Path(output_dir) / 'meta_data.json', "w") as f:
    json.dump(meta_data, f)

In [4]:
df_merged = pd.read_csv(Path(output_dir) / "merged_dataset.csv")
df_merged.to_parquet(Path(output_dir) / "merged_dataset.parquet", index=False)

C:\Users\jonas\AppData\Local\Temp\ipykernel_19392\4035431215.py:1: DtypeWarning: Columns (0: NEUTRALDESCRIPTION) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged = pd.read_csv(Path(output_dir) / "merged_dataset.csv")
